# Metrics

## 1. Object Detection and Localization Metrics

These metrics evaluate how well a model detects and localizes objects (vehicles, pedestrians, cyclists) in 2D images or 3D point clouds.

| Metric | What it measures | Typical use |
|--------|-----------------|-------------|
| **IoU** (Intersection over Union) | Overlap between predicted and ground-truth box | Threshold for matching predictions to GT |
| **Precision / Recall** | Trade-off between false positives and missed detections | Per-class evaluation |
| **AP** (Average Precision) | Area under the Precision-Recall curve | Per-class summary |
| **mAP** | Mean AP across all classes | Overall detector quality |
| **APH** (AP weighted by Heading) | AP penalized by heading error (Waymo-specific) | 3D detection with orientation |
| **NDS** (nuScenes Detection Score) | Weighted combination of mAP + TP errors | nuScenes benchmark |
| **BEV AP** | AP evaluated in Bird's Eye View | 3D detection projected to ground plane |

### Key concepts
- **True Positive (TP):** IoU ≥ threshold (commonly 0.5 or 0.7)
- **False Positive (FP):** prediction with no matching GT (IoU < threshold)
- **False Negative (FN):** GT with no matching prediction
- **Confidence threshold:** predictions are ranked by confidence; PR curve sweeps over thresholds

$P = \frac{TP}{TP + FP}, \quad R = \frac{TP}{TP + FN}$

In [11]:
import numpy as np
from typing import List, Tuple

# ============================================================
# 1a. Intersection over Union (IoU) — 2D axis-aligned boxes
# ============================================================
def iou_2d(box_a: np.ndarray, box_b: np.ndarray) -> float:
    """
    Compute IoU between two 2D boxes.
    Each box: [x_min, y_min, x_max, y_max]
    """
    x1 = max(box_a[0], box_b[0])
    y1 = max(box_a[1], box_b[1])
    x2 = min(box_a[2], box_b[2])
    y2 = min(box_a[3], box_b[3])
    
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area_a = (box_a[2] - box_a[0]) * (box_a[3] - box_a[1])
    area_b = (box_b[2] - box_b[0]) * (box_b[3] - box_b[1])
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0


# Quick sanity check
box1 = np.array([0, 0, 2, 2])
box2 = np.array([1, 1, 3, 3])
print(f"IoU(box1, box2) = {iou_2d(box1, box2):.4f}")  # expect 1/7 ≈ 0.1429

IoU(box1, box2) = 0.1429


In [12]:
# ============================================================
# 1b. 3D IoU — axis-aligned 3D bounding boxes
# ============================================================
def iou_3d_aabb(box_a: np.ndarray, box_b: np.ndarray) -> float:
    """
    Compute IoU for two 3D axis-aligned bounding boxes.
    Each box: [x_min, y_min, z_min, x_max, y_max, z_max]
    """
    # Intersection along each axis
    x1 = max(box_a[0], box_b[0]); x2 = min(box_a[3], box_b[3])
    y1 = max(box_a[1], box_b[1]); y2 = min(box_a[4], box_b[4])
    z1 = max(box_a[2], box_b[2]); z2 = min(box_a[5], box_b[5])
    
    inter = max(0, x2 - x1) * max(0, y2 - y1) * max(0, z2 - z1)
    vol_a = (box_a[3]-box_a[0]) * (box_a[4]-box_a[1]) * (box_a[5]-box_a[2])
    vol_b = (box_b[3]-box_b[0]) * (box_b[4]-box_b[1]) * (box_b[5]-box_b[2])
    union = vol_a + vol_b - inter
    return inter / union if union > 0 else 0.0


box3d_a = np.array([0, 0, 0, 2, 2, 2])
box3d_b = np.array([1, 1, 1, 3, 3, 3])
print(f"3D IoU = {iou_3d_aabb(box3d_a, box3d_b):.4f}")  # expect 1/15 ≈ 0.0667

3D IoU = 0.0667


In [ ]:
# ============================================================
# 1c. Precision, Recall, and Average Precision (AP)
# ============================================================
def compute_precision_recall(
    pred_boxes: List[np.ndarray],
    pred_scores: List[float],
    gt_boxes: List[np.ndarray],
    iou_threshold: float = 0.5
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Compute precision-recall curve for a single class in one frame.
    pred_boxes:  list of predicted boxes [x1,y1,x2,y2]
    pred_scores: confidence scores
    gt_boxes:    list of ground-truth boxes
    
    
    Sorted predictions (by confidence):
    i=0: score=0.95, TP  →  cumTP=1, cumFP=0  →  P=1/1=1.00, R=1/3=0.33
    i=1: score=0.90, TP  →  cumTP=2, cumFP=0  →  P=2/2=1.00, R=2/3=0.67
    i=2: score=0.80, FP  →  cumTP=2, cumFP=1  →  P=2/3=0.67, R=2/3=0.67
    i=3: score=0.70, TP  →  cumTP=3, cumFP=1  →  P=3/4=0.75, R=3/3=1.00

    """
    # Sort predictions by descending confidence
    sorted_idx = np.argsort(pred_scores)[::-1]
    pred_boxes = [pred_boxes[i] for i in sorted_idx]
    
    tp = np.zeros(len(pred_boxes))
    fp = np.zeros(len(pred_boxes))
    matched_gt = set()
    
    for i, pred in enumerate(pred_boxes):
        best_iou = 0.0
        best_gt_idx = -1
        for j, gt in enumerate(gt_boxes):
            iou = iou_2d(np.array(pred), np.array(gt))
            if iou > best_iou:
                best_iou = iou
                best_gt_idx = j
        if best_iou >= iou_threshold and best_gt_idx not in matched_gt:
            tp[i] = 1
            matched_gt.add(best_gt_idx)
        else:
            fp[i] = 1
        print(tp, fp)
    tp_cumsum = np.cumsum(tp)
    fp_cumsum = np.cumsum(fp)
    precision = tp_cumsum / (tp_cumsum + fp_cumsum)
    recall = tp_cumsum / len(gt_boxes) if len(gt_boxes) > 0 else tp_cumsum
    return precision, recall


def compute_ap(precision: np.ndarray, recall: np.ndarray) -> float:
    """
    Compute AP using the 11-point interpolation (PASCAL VOC style)
    or all-point interpolation (COCO style).
    Here: all-point interpolation (area under smoothed PR curve).
    """
    # Prepend sentinel values
    recall = np.concatenate(([0.0], recall, [1.0]))
    precision = np.concatenate(([1.0], precision, [0.0]))
    
    # Smooth precision: make it monotonically decreasing
    for i in range(len(precision) - 2, -1, -1):
        precision[i] = max(precision[i], precision[i + 1])
    
    # Find points where recall changes
    change_points = np.where(recall[1:] != recall[:-1])[0]
    ap = np.sum((recall[change_points + 1] - recall[change_points]) * precision[change_points + 1])
    return ap


# --- Demo ---
gt = [np.array([10, 10, 50, 50]), np.array([60, 60, 100, 100]), np.array([120, 120, 160, 160])]
preds = [np.array([12, 12, 48, 48]), np.array([62, 58, 98, 102]),
         np.array([200, 200, 240, 240]), np.array([122, 118, 158, 162])]
scores = [0.95, 0.90, 0.80, 0.70]

prec, rec = compute_precision_recall(preds, scores, gt, iou_threshold=0.5)
ap = compute_ap(prec, rec)
print(f"Precision: {prec}")
print(f"Recall:    {rec}")
print(f"AP@0.5   = {ap:.4f}")

[1. 0. 0. 0.] [0. 0. 0. 0.]
[1. 1. 0. 0.] [0. 0. 0. 0.]
[1. 1. 0. 0.] [0. 0. 1. 0.]
[1. 1. 0. 1.] [0. 0. 1. 0.]
Precision: [1.         1.         0.66666667 0.75      ]
Recall:    [0.33333333 0.66666667 0.66666667 1.        ]
AP@0.5   = 0.9167


### Non-Maximum Suppression (NMS)

NMS is a **post-processing** step applied to raw detector outputs to remove redundant overlapping boxes for the same object.

**Algorithm:**
1. Sort all detections by confidence score (descending)
2. Pick the highest-confidence box → keep it
3. Compute IoU of all remaining boxes with this box
4. Remove any box with IoU ≥ threshold (they're duplicates)
5. Repeat from step 2 with remaining boxes

**Variants:**
| Variant | Difference |
|---------|-----------|
| **Hard NMS** | Binary suppress if IoU ≥ threshold (standard) |
| **Soft NMS** | Decay confidence instead of removing: $s_i \leftarrow s_i \cdot e^{-\text{IoU}^2 / \sigma}$ |
| **Batched NMS** | Run NMS independently per class |
| **Matrix NMS** | Vectorized approximation (used in SOLO/SOLOv2) |

In [23]:
# ============================================================
# 1d. Non-Maximum Suppression (NMS)
# ============================================================
def nms(boxes: np.ndarray, scores: np.ndarray, iou_threshold: float = 0.5) -> List[int]:
    """
    Hard NMS: remove overlapping boxes, keep highest-confidence ones.
    boxes:  (N, 4) array of [x1, y1, x2, y2]
    scores: (N,) confidence scores
    Returns: list of indices to keep
    """
    order = np.argsort(scores)[::-1]  # sort by confidence desc
    keep = []
    
    while len(order) > 0:
        # Pick the best remaining box
        i = order[0]
        keep.append(i)
        
        if len(order) == 1:
            break
        
        # Compute IoU of this box with all remaining boxes
        remaining = order[1:]
        ious = np.array([iou_2d(boxes[i], boxes[j]) for j in remaining])
        
        # Keep only boxes with IoU < threshold (not duplicates)
        mask = ious < iou_threshold
        order = remaining[mask]
    
    return keep


def soft_nms(boxes: np.ndarray, scores: np.ndarray,
             iou_threshold: float = 0.5, sigma: float = 0.5,
             score_threshold: float = 0.01) -> Tuple[List[int], np.ndarray]:
    """
    Soft NMS: instead of removing overlapping boxes, decay their scores.
    s_i = s_i * exp(-IoU^2 / sigma)
    Returns: (kept indices, updated scores)
    """
    scores = scores.copy()
    N = len(boxes)
    indices = list(range(N))
    keep = []
    
    while len(indices) > 0:
        # Find best score among remaining
        idx_in_remaining = np.argmax(scores[indices])
        best = indices[idx_in_remaining]
        keep.append(best)
        indices.pop(idx_in_remaining)
        
        # Decay scores of overlapping boxes
        to_remove = []
        for pos, j in enumerate(indices):
            iou = iou_2d(boxes[best], boxes[j])
            # Gaussian decay
            scores[j] *= np.exp(-(iou ** 2) / sigma)
            if scores[j] < score_threshold:
                to_remove.append(pos)
        
        # Remove boxes whose score fell below threshold
        for pos in sorted(to_remove, reverse=True):
            indices.pop(pos)
    
    return keep, scores


# --- Demo: Hard NMS vs Soft NMS ---
# 5 boxes, 3 of which heavily overlap (detecting the same car)
boxes = np.array([
    [10, 10, 50, 50],   # car A — best
    [12, 11, 48, 49],   # car A — duplicate
    [14, 12, 52, 51],   # car A — duplicate
    [100, 100, 140, 140], # car B
    [200, 200, 240, 240], # car C
], dtype=float)
scores = np.array([0.95, 0.88, 0.82, 0.90, 0.75])

# Hard NMS
kept_hard = nms(boxes, scores, iou_threshold=0.5)
print(f"Hard NMS kept indices: {kept_hard}")
print(f"  → {len(kept_hard)} boxes (from {len(boxes)})")

# Soft NMS
kept_soft, new_scores = soft_nms(boxes, scores, sigma=0.5)
print(f"\nSoft NMS kept indices: {kept_soft}")
print(f"  Updated scores: {[f'{new_scores[i]:.3f}' for i in kept_soft]}")

Hard NMS kept indices: [0, 3, 4]
  → 3 boxes (from 5)

Soft NMS kept indices: [0, 3, 4, 2, 1]
  Updated scores: ['0.950', '0.900', '0.750', '0.229', '0.059']


### Waymo-Specific: APH (Average Precision weighted by Heading)

In the **Waymo Open Dataset**, standard AP is extended with heading accuracy:

$$\text{APH} = \text{AP} \times \frac{1}{|\text{TP}|} \sum_{i \in \text{TP}} \max\left(0,\; \cos(\Delta\theta_i)\right)$$

where $\Delta\theta_i$ is the heading error between prediction and ground truth.

- **LEVEL 1:** objects with ≥ 5 LiDAR points (easy)
- **LEVEL 2:** objects with ≥ 1 LiDAR point (includes hard cases)

### mAP Variants

| Variant | Description |
|---------|-------------|
| **mAP@0.5** | IoU threshold = 0.5 (PASCAL VOC) |
| **mAP@0.75** | Stricter threshold |
| **mAP@[0.5:0.95]** | Average over IoU thresholds from 0.5 to 0.95 in steps of 0.05 (COCO-style) |
| **BEV mAP** | mAP computed in Bird's Eye View (x-y plane only, ignores height) |
| **3D mAP** | Full 3D IoU-based mAP |

### nuScenes Detection Score (NDS)

$$\text{NDS} = \frac{1}{10}\left[5 \cdot \text{mAP} + \sum_{\text{TP metric}} (1 - \min(1, \text{err}))\right]$$

TP metrics: **ATE** (translation), **ASE** (scale), **AOE** (orientation), **AVE** (velocity), **AAE** (attribute)

## 2. Object Tracking Metrics (MOT)

Multi-Object Tracking (MOT) metrics evaluate how well a system maintains consistent identities for detected objects over time.

| Metric | Formula / Idea | What it captures |
|--------|---------------|-----------------|
| **MOTA** | $1 - \frac{\text{FN} + \text{FP} + \text{IDSW}}{\text{GT}}$ | Overall tracking accuracy (can be negative) |
| **MOTP** | $\frac{\sum_{t,i} d_{t,i}}{\sum_t c_t}$ | Average localization error of matched pairs |
| **IDF1** | $\frac{2 \cdot \text{IDTP}}{2 \cdot \text{IDTP} + \text{IDFP} + \text{IDFN}}$ | Identity-aware F1 score |
| **HOTA** | $\sqrt{\text{DetA} \times \text{AssA}}$ | Balanced detection + association |
| **ID Switches** | Count of identity changes | Track consistency |
| **Frag** | Count of track fragmentations | Track continuity |
| **MT / ML** | Mostly Tracked / Mostly Lost ratios | Track completeness |

### MOTA vs HOTA

- **MOTA** dominated MOT benchmarks but over-weights detection (FP/FN dominate the formula).
- **HOTA** (Higher Order Tracking Accuracy) balances **detection accuracy (DetA)** and **association accuracy (AssA)** via geometric mean, and is averaged over IoU thresholds like COCO mAP.

### Waymo Tracking Metrics
Waymo uses **MOTA** and **MOTP** at LEVEL 1 and LEVEL 2, plus tracking-specific **APH** that considers temporal consistency.

In [24]:
# ============================================================
# 2a. MOTA & MOTP
# ============================================================
def compute_mota(num_gt: int, num_fp: int, num_fn: int, num_id_switches: int) -> float:
    """
    Multi-Object Tracking Accuracy.
    MOTA = 1 - (FN + FP + IDSW) / GT
    Can be negative if errors exceed GT count.
    """
    if num_gt == 0:
        return 0.0
    return 1.0 - (num_fn + num_fp + num_id_switches) / num_gt


def compute_motp(distances: List[float]) -> float:
    """
    Multi-Object Tracking Precision.
    Average distance/IoU between matched prediction-GT pairs.
    Lower is better when using distance; higher is better when using IoU.
    """
    if len(distances) == 0:
        return 0.0
    return np.mean(distances)


# Example: 100 GT objects across all frames
# 10 false positives, 15 misses, 5 identity switches
mota = compute_mota(num_gt=100, num_fp=10, num_fn=15, num_id_switches=5)
print(f"MOTA = {mota:.2f}")  # 1 - 30/100 = 0.70

# Average IoU of matched pairs
motp = compute_motp([0.85, 0.78, 0.92, 0.70, 0.88])
print(f"MOTP (avg IoU) = {motp:.4f}")

MOTA = 0.70
MOTP (avg IoU) = 0.8260


In [25]:
# ============================================================
# 2b. IDF1 — Identity F1 Score
# ============================================================
def compute_idf1(id_tp: int, id_fp: int, id_fn: int) -> float:
    """
    IDF1 = 2 * IDTP / (2 * IDTP + IDFP + IDFN)
    
    IDTP: number of frames where a predicted ID correctly matches a GT ID
    IDFP: predicted ID frames with no matching GT ID
    IDFN: GT ID frames with no matching predicted ID
    
    IDF1 focuses on "how long do we correctly maintain an identity?"
    """
    denom = 2 * id_tp + id_fp + id_fn
    if denom == 0:
        return 0.0
    return 2 * id_tp / denom


# Example
idf1 = compute_idf1(id_tp=80, id_fp=10, id_fn=15)
print(f"IDF1 = {idf1:.4f}")


# ============================================================
# 2c. HOTA — Higher Order Tracking Accuracy
# ============================================================
def compute_hota(detection_accuracy: float, association_accuracy: float) -> float:
    """
    HOTA = sqrt(DetA * AssA)
    
    DetA (Detection Accuracy):  Jaccard index of detection matching
    AssA (Association Accuracy): Jaccard index of association (identity) matching
    
    Both DetA and AssA are computed per IoU threshold, then averaged.
    """
    return np.sqrt(detection_accuracy * association_accuracy)


# Example
det_a = 0.75  # 75% detection accuracy
ass_a = 0.80  # 80% association accuracy
hota = compute_hota(det_a, ass_a)
print(f"HOTA = {hota:.4f}")  # sqrt(0.75 * 0.80) ≈ 0.7746

IDF1 = 0.8649
HOTA = 0.7746


## 3. Motion Prediction Metrics

Motion prediction (behavior prediction) forecasts future trajectories of agents. Models typically output **K possible trajectories** with associated probabilities.

| Metric | Formula | What it captures |
|--------|---------|-----------------|
| **ADE** | $\frac{1}{T}\sum_{t=1}^{T} \|\hat{p}_t - p_t\|_2$ | Avg displacement error over the trajectory |
| **FDE** | $\|\hat{p}_T - p_T\|_2$ | Displacement error at the final timestep |
| **minADE$_K$** | $\min_{k=1..K} \text{ADE}(k)$ | Best-of-K average error (multi-modal) |
| **minFDE$_K$** | $\min_{k=1..K} \text{FDE}(k)$ | Best-of-K final error |
| **Miss Rate** | $\frac{1}{N}\sum \mathbb{1}[\text{FDE} > \delta]$ | Fraction of samples where FDE exceeds threshold $\delta$ |
| **Brier-minFDE** | $\text{minFDE} + (1 - p_{\text{best}})^2$ | Penalizes low probability assigned to best trajectory |
| **mAP (Waymo)** | Motion-prediction mAP | Waymo's metric treating prediction as a detection problem |

### Key design choices
- **Horizon:** typically 3s, 5s, or 8s into the future
- **Multi-modality:** minADE/minFDE with K=6 is standard (Waymo uses K=6)
- **Brier-minFDE** encourages calibrated confidence scores alongside accurate trajectories

In [26]:
# ============================================================
# 3a. ADE, FDE, minADE, minFDE
# ============================================================
def compute_ade(pred_traj: np.ndarray, gt_traj: np.ndarray) -> float:
    """
    Average Displacement Error.
    pred_traj, gt_traj: shape (T, 2) — x,y positions over T timesteps.
    """
    return np.mean(np.linalg.norm(pred_traj - gt_traj, axis=1))


def compute_fde(pred_traj: np.ndarray, gt_traj: np.ndarray) -> float:
    """
    Final Displacement Error.
    """
    return np.linalg.norm(pred_traj[-1] - gt_traj[-1])


def compute_min_ade_fde(
    pred_trajs: np.ndarray, gt_traj: np.ndarray
) -> Tuple[float, float, int]:
    """
    min ADE_K and min FDE_K over K predicted trajectories.
    pred_trajs: shape (K, T, 2)
    gt_traj:    shape (T, 2)
    Returns: (minADE, minFDE, best_k_index)
    """
    K = pred_trajs.shape[0]
    ades = [compute_ade(pred_trajs[k], gt_traj) for k in range(K)]
    fdes = [compute_fde(pred_trajs[k], gt_traj) for k in range(K)]
    
    best_ade_idx = int(np.argmin(ades))
    best_fde_idx = int(np.argmin(fdes))
    
    return ades[best_ade_idx], fdes[best_fde_idx], best_fde_idx


# --- Demo ---
np.random.seed(42)
T = 30  # 3 seconds at 10Hz
gt = np.cumsum(np.random.randn(T, 2) * 0.3, axis=0)  # random walk GT

K = 6
preds = np.stack([gt + np.random.randn(T, 2) * (0.5 + k * 0.3) for k in range(K)])

for k in range(K):
    print(f"  Trajectory {k}: ADE={compute_ade(preds[k], gt):.3f}, FDE={compute_fde(preds[k], gt):.3f}")

min_ade, min_fde, best_k = compute_min_ade_fde(preds, gt)
print(f"\nminADE_6 = {min_ade:.3f}")
print(f"minFDE_6 = {min_fde:.3f} (best trajectory: {best_k})")

  Trajectory 0: ADE=0.574, FDE=0.684
  Trajectory 1: ADE=1.027, FDE=2.186
  Trajectory 2: ADE=1.380, FDE=1.187
  Trajectory 3: ADE=1.785, FDE=1.439
  Trajectory 4: ADE=1.638, FDE=1.510
  Trajectory 5: ADE=2.506, FDE=2.362

minADE_6 = 0.574
minFDE_6 = 0.684 (best trajectory: 0)


In [27]:
# ============================================================
# 3b. Miss Rate & Brier-minFDE
# ============================================================
def compute_miss_rate(
    pred_trajs: np.ndarray, gt_traj: np.ndarray, threshold: float = 2.0
) -> float:
    """
    Miss Rate: fraction of K trajectories whose FDE exceeds threshold.
    For a dataset, this is the fraction of *samples* where the best 
    trajectory still misses.
    """
    fdes = [compute_fde(pred_trajs[k], gt_traj) for k in range(pred_trajs.shape[0])]
    min_fde = min(fdes)
    return 1.0 if min_fde > threshold else 0.0


def compute_brier_min_fde(
    pred_trajs: np.ndarray,
    pred_probs: np.ndarray,
    gt_traj: np.ndarray
) -> float:
    """
    Brier-minFDE = minFDE + (1 - p_best)^2
    
    Encourages the model to assign high probability to its best trajectory.
    pred_probs: shape (K,), probabilities summing to 1.
    """
    K = pred_trajs.shape[0]
    fdes = [compute_fde(pred_trajs[k], gt_traj) for k in range(K)]
    best_k = int(np.argmin(fdes))
    min_fde = fdes[best_k]
    brier_penalty = (1.0 - pred_probs[best_k]) ** 2
    return min_fde + brier_penalty


# --- Demo ---
probs = np.array([0.4, 0.25, 0.15, 0.10, 0.05, 0.05])  # K=6 trajectory probs
brier = compute_brier_min_fde(preds, probs, gt)
miss = compute_miss_rate(preds, gt, threshold=2.0)
print(f"Brier-minFDE = {brier:.3f}")
print(f"Miss Rate (δ=2.0) = {miss}")

Brier-minFDE = 1.044
Miss Rate (δ=2.0) = 0.0


## 4. Planning Metrics

Planning metrics evaluate the quality of the ego-vehicle's planned trajectory.

| Metric | What it measures |
|--------|-----------------|
| **L2 Error** | Euclidean distance between planned and expert trajectory at each timestep |
| **Collision Rate** | % of planned trajectories that collide with other agents |
| **Off-road Rate** | % of trajectories that leave drivable area |
| **Comfort Metrics** | Jerk, lateral acceleration — smoothness of the plan |
| **Progress** | Distance traveled toward goal (avoid overly cautious plans) |
| **Time-to-Collision (TTC)** | Minimum time until a collision would occur at current velocity |

### Closed-loop vs Open-loop Planning

| | Open-loop | Closed-loop |
|---|----------|-------------|
| **How** | Compare planned trajectory to recorded expert | Run planner in simulation, environment reacts |
| **Pros** | Fast, uses existing logs | Realistic; captures compounding errors |
| **Cons** | Distributional shift; doesn't test reactions | Expensive; simulator fidelity matters |
| **Metrics** | L2 error, ADE/FDE vs expert | Collision rate, completion rate, comfort |

In [28]:
# ============================================================
# 4a. Planning metrics: L2, Collision Rate, Comfort
# ============================================================
def compute_planning_l2(
    planned: np.ndarray, expert: np.ndarray, horizons: List[int] = [10, 30, 50]
) -> dict:
    """
    L2 error between planned and expert trajectory at specific time horizons.
    planned, expert: shape (T, 2)
    horizons: list of timestep indices (e.g., 10=1s, 30=3s, 50=5s at 10Hz)
    """
    results = {}
    for h in horizons:
        if h < len(planned) and h < len(expert):
            l2 = np.linalg.norm(planned[h] - expert[h])
            results[f"L2@{h}"] = l2
    return results


def check_collision(
    ego_traj: np.ndarray,
    agent_trajs: List[np.ndarray],
    collision_radius: float = 2.0
) -> Tuple[bool, int]:
    """
    Simple collision check: does the ego come within `collision_radius` 
    of any agent at any timestep?
    ego_traj: shape (T, 2)
    agent_trajs: list of (T, 2) arrays
    """
    T = ego_traj.shape[0]
    for t in range(T):
        for agent in agent_trajs:
            if t < len(agent):
                dist = np.linalg.norm(ego_traj[t] - agent[t])
                if dist < collision_radius:
                    return True, t
    return False, -1


def compute_jerk(trajectory: np.ndarray, dt: float = 0.1) -> float:
    """
    Compute average jerk (rate of change of acceleration) as a comfort metric.
    trajectory: shape (T, 2)
    dt: time between steps
    """
    velocity = np.diff(trajectory, axis=0) / dt
    acceleration = np.diff(velocity, axis=0) / dt
    jerk = np.diff(acceleration, axis=0) / dt
    return np.mean(np.linalg.norm(jerk, axis=1))


# --- Demo ---
np.random.seed(0)
T = 50
expert_traj = np.column_stack([np.linspace(0, 50, T), np.zeros(T)])  # straight line
planned_traj = expert_traj + np.random.randn(T, 2) * 0.5  # noisy plan

l2_results = compute_planning_l2(planned_traj, expert_traj)
print("Planning L2 errors:", {k: f"{v:.3f}" for k, v in l2_results.items()})

# Collision check
agent1 = np.column_stack([np.linspace(25, 25, T), np.linspace(-5, 5, T)])
collides, at_t = check_collision(planned_traj, [agent1], collision_radius=2.0)
print(f"Collision: {collides}" + (f" at t={at_t}" if collides else ""))

# Comfort
jerk_val = compute_jerk(planned_traj)
print(f"Average jerk = {jerk_val:.2f} m/s³")

Planning L2 errors: {'L2@10': '1.318', 'L2@30': '0.381'}
Collision: True at t=24
Average jerk = 2435.48 m/s³


## 5. Segmentation Metrics

Semantic / panoptic segmentation is used for scene understanding (drivable area, lane markings, sidewalks).

| Metric | Formula | Use |
|--------|---------|-----|
| **mIoU** | $\frac{1}{C}\sum_{c=1}^{C} \frac{TP_c}{TP_c + FP_c + FN_c}$ | Mean IoU across all classes |
| **Pixel Accuracy** | $\frac{\text{correct pixels}}{\text{total pixels}}$ | Simple but biased toward dominant class |
| **PQ (Panoptic Quality)** | $\text{SQ} \times \text{RQ}$ | Panoptic segmentation = semantic + instance |
| **SQ (Segmentation Quality)** | $\frac{\sum \text{IoU}(p,g)}{|\text{TP}|}$ | Average IoU of matched segments |
| **RQ (Recognition Quality)** | $\frac{|\text{TP}|}{|\text{TP}| + \frac{1}{2}|\text{FP}| + \frac{1}{2}|\text{FN}|}$ | F1-like score for segment matching |

### Panoptic Quality Breakdown

$$\text{PQ} = \underbrace{\frac{\sum_{(p,g) \in \text{TP}} \text{IoU}(p,g)}{|\text{TP}|}}_{\text{SQ}} \times \underbrace{\frac{|\text{TP}|}{|\text{TP}| + \frac{1}{2}|\text{FP}| + \frac{1}{2}|\text{FN}|}}_{\text{RQ}}$$

In [29]:
# ============================================================
# 5a. mIoU for Semantic Segmentation
# ============================================================
def compute_miou(pred: np.ndarray, gt: np.ndarray, num_classes: int) -> Tuple[float, np.ndarray]:
    """
    Compute mean Intersection over Union for semantic segmentation.
    pred, gt: (H, W) integer arrays with class labels
    """
    ious = []
    for c in range(num_classes):
        pred_c = (pred == c)
        gt_c = (gt == c)
        intersection = np.sum(pred_c & gt_c)
        union = np.sum(pred_c | gt_c)
        if union == 0:
            ious.append(float('nan'))  # class not present
        else:
            ious.append(intersection / union)
    
    ious = np.array(ious)
    miou = np.nanmean(ious)
    return miou, ious


# --- Demo ---
np.random.seed(42)
H, W, C = 64, 64, 5  # 5 classes: road, sidewalk, vehicle, pedestrian, background
gt_seg = np.random.randint(0, C, (H, W))
pred_seg = gt_seg.copy()
# Add some noise
noise_mask = np.random.rand(H, W) < 0.15  # 15% error
pred_seg[noise_mask] = np.random.randint(0, C, noise_mask.sum())

miou, per_class = compute_miou(pred_seg, gt_seg, C)
class_names = ['road', 'sidewalk', 'vehicle', 'pedestrian', 'background']
for name, iou in zip(class_names, per_class):
    print(f"  {name:12s}: IoU = {iou:.4f}")
print(f"\nmIoU = {miou:.4f}")


# ============================================================
# 5b. Panoptic Quality (PQ = SQ × RQ)
# ============================================================
def compute_panoptic_quality(
    matched_ious: List[float], num_fp: int, num_fn: int
) -> Tuple[float, float, float]:
    """
    PQ = SQ × RQ
    matched_ious: IoU values for each TP matched pair
    """
    tp = len(matched_ious)
    sq = np.mean(matched_ious) if tp > 0 else 0.0
    rq = tp / (tp + 0.5 * num_fp + 0.5 * num_fn) if (tp + num_fp + num_fn) > 0 else 0.0
    pq = sq * rq
    return pq, sq, rq


pq, sq, rq = compute_panoptic_quality([0.85, 0.78, 0.92, 0.70], num_fp=2, num_fn=1)
print(f"\nPQ = {pq:.4f}, SQ = {sq:.4f}, RQ = {rq:.4f}")

  road        : IoU = 0.7693
  sidewalk    : IoU = 0.7719
  vehicle     : IoU = 0.7785
  pedestrian  : IoU = 0.8013
  background  : IoU = 0.7758

mIoU = 0.7794

PQ = 0.5909, SQ = 0.8125, RQ = 0.7273


## 6. Calibration & Uncertainty Metrics

Self-driving models must know **when they don't know**. Calibration measures whether predicted confidences match actual accuracy.

| Metric | Formula | Purpose |
|--------|---------|---------|
| **ECE** (Expected Calibration Error) | $\sum_{b=1}^{B} \frac{n_b}{N} |\text{acc}(b) - \text{conf}(b)|$ | Gap between confidence and accuracy |
| **Brier Score** | $\frac{1}{N}\sum (p_i - y_i)^2$ | Mean squared error of predicted probabilities |
| **NLL** (Negative Log-Likelihood) | $-\frac{1}{N}\sum \log p(y_i)$ | Probabilistic calibration |

### Why calibration matters in AV
- A detector saying "90% sure this is a pedestrian" should be correct ~90% of the time
- Overconfident models make dangerous planning decisions
- Underconfident models cause unnecessary braking / hesitation

In [30]:
# ============================================================
# 6a. Expected Calibration Error (ECE)
# ============================================================
def compute_ece(confidences: np.ndarray, accuracies: np.ndarray, num_bins: int = 10) -> float:
    """
    Expected Calibration Error.
    confidences: predicted confidence for each sample
    accuracies:  1 if correct, 0 if incorrect for each sample
    """
    bin_boundaries = np.linspace(0, 1, num_bins + 1)
    ece = 0.0
    for i in range(num_bins):
        mask = (confidences > bin_boundaries[i]) & (confidences <= bin_boundaries[i + 1])
        if mask.sum() == 0:
            continue
        bin_acc = accuracies[mask].mean()
        bin_conf = confidences[mask].mean()
        ece += mask.sum() / len(confidences) * abs(bin_acc - bin_conf)
    return ece


def compute_brier_score(probs: np.ndarray, labels: np.ndarray) -> float:
    """Brier Score = mean((p - y)^2)"""
    return np.mean((probs - labels) ** 2)


# --- Demo ---
np.random.seed(42)
N = 1000
# Well-calibrated model
true_labels = np.random.randint(0, 2, N)
good_confs = np.clip(true_labels * 0.7 + np.random.rand(N) * 0.3, 0, 1)
good_preds = (good_confs > 0.5).astype(int)
good_correct = (good_preds == true_labels).astype(float)

# Overconfident model
bad_confs = np.clip(np.random.rand(N) * 0.3 + 0.7, 0, 1)  # always high conf
bad_preds = np.random.randint(0, 2, N)
bad_correct = (bad_preds == true_labels).astype(float)

print("Well-calibrated model:")
print(f"  ECE   = {compute_ece(good_confs, good_correct):.4f}")
print(f"  Brier = {compute_brier_score(good_confs, true_labels.astype(float)):.4f}")

print("\nOverconfident model:")
print(f"  ECE   = {compute_ece(bad_confs, bad_correct):.4f}")
print(f"  Brier = {compute_brier_score(bad_confs, true_labels.astype(float)):.4f}")

Well-calibrated model:
  ECE   = 0.4931
  Brier = 0.0294

Overconfident model:
  ECE   = 0.3854
  Brier = 0.3715


## 7. System-Level & Latency Metrics

In production self-driving, model quality is necessary but not sufficient. System-level constraints matter.

| Metric | What it measures | Target |
|--------|-----------------|--------|
| **Latency (p50/p99)** | Inference time per frame | < 100ms for real-time |
| **Throughput** | Frames per second | ≥ 10 FPS (LiDAR rate) |
| **Memory** | GPU/CPU memory usage | Fits on-vehicle compute |
| **Disengagement Rate** | Human takeover frequency per mile | Lower = better |
| **Miles per Intervention** | Distance between safety-critical events | Higher = better |

### Interview Tips 🎯

1. **Always clarify the task** before jumping to metrics: detection? tracking? prediction? planning?
2. **Know IoU thresholds**: 0.5 is lenient; 0.7 is standard for vehicles in Waymo/nuScenes
3. **Understand multi-modality**: minADE/minFDE with K trajectories for motion prediction
4. **Waymo-specific**: APH (heading-weighted AP), LEVEL 1 vs LEVEL 2 difficulty
5. **Open-loop vs Closed-loop**: know the trade-offs for planning evaluation
6. **Calibration matters**: AV models must be calibrated, not just accurate
7. **System constraints**: latency matters as much as accuracy in production

## 8. Quick Reference — Metrics Cheat Sheet

| Module | Primary Metrics | Waymo-Specific |
|--------|----------------|----------------|
| **2D Detection** | mAP@[0.5:0.95], AP per class | — |
| **3D Detection** | 3D mAP, BEV mAP | APH (L1/L2) |
| **Tracking** | MOTA, MOTP, HOTA, IDF1 | MOTA (L1/L2) |
| **Motion Prediction** | minADE₆, minFDE₆, Miss Rate | Brier-minFDE, Soft mAP |
| **Planning** | L2 error, Collision Rate, Comfort | Closed-loop sim |
| **Segmentation** | mIoU, PQ | Panoptic seg |
| **Calibration** | ECE, Brier Score | — |
| **System** | Latency p99, FPS, Memory | Disengagements/mile |